# 🚀 Huấn Luyện Toàn Bộ 6 Cấu Hình Ablation Study D2SA Trên NVIDIA A100 (40GB VRAM)

**Dự án:** Amodal Shape Prediction (Swin-UNet + SAM 2.1) — Mở rộng sang tập dữ liệu **MVTec D2SA**  
**Môi trường:** Google Colab Pro / Pro+ với GPU **NVIDIA A100 (40GB VRAM)**  
**Tính năng đặc biệt:**  
- 🔄 **Lưu trực tiếp vào Google Drive**: Toàn bộ Checkpoints (`.pth`), Text Logs (`.log`), Metrics (`.jsonl`), và Kết quả đánh giá (`.json`) được ghi thẳng vào Google Drive sau mỗi epoch với cơ chế `flush` + `fsync` nguyên tử.
- ⚡ **Tối ưu hóa A100 40GB**: Huấn luyện với `batch_size=16`, `accumulation_steps=1`, `num_workers=8`. VRAM tiêu thụ ~12-13 GB (chỉ chiếm ~30% VRAM của A100 40GB, loại bỏ 100% rủi ro OOM).
- 📊 **Tự động tổng hợp Table V**: Sau khi hoàn thành, tự động xuất bảng so sánh 6 cấu hình (mIoU, Dice, Precision, Recall, Invisible mIoU) chuẩn Markdown.

---

### 📌 Bước 1: Kiểm Tra GPU & Thông Số Phần Cứng
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 GPU -> High-RAM**.

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Name: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")

### 📁 Bước 2: Kết Nối Google Drive Để Lưu Trực Tiếp
Toàn bộ checkpoints và kết quả sẽ được lưu liên tục vào Google Drive để không bị mất khi Colab ngắt kết nối.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Thiết lập thư mục lưu trữ trên Google Drive của bạn
DRIVE_DIR = "/content/drive/MyDrive/amodal_shape_prediction"
os.makedirs(f"{DRIVE_DIR}/checkpoints/d2sa", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs/d2sa", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results/d2sa", exist_ok=True)

print(f"✅ Đã kết nối Google Drive! Thư mục lưu trữ: {DRIVE_DIR}")

### 📥 Bước 3: Chuẩn Bị Mã Nguồn & Cài Đặt Thư Viện
Tự động nhận diện: Nếu bạn đã upload thư mục `amodal_shape_prediction` lên Google Drive thì copy sang SSD Colab để nạp code siêu nhanh, nếu chưa thì tự động clone từ GitHub.

In [ ]:
import os

DRIVE_REPO = "/content/drive/MyDrive/amodal_shape_prediction"
LOCAL_REPO = "/content/amodal_shape_prediction"

# Nếu đã upload thư mục lên Google Drive:
if os.path.exists(os.path.join(DRIVE_REPO, "scripts")):
    print("📁 Phát hiện repo có sẵn trên Google Drive! Sao chép sang SSD Colab...")
    !cp -r {DRIVE_REPO} {LOCAL_REPO}
    %cd {LOCAL_REPO}
# Nếu chưa có trên Drive -> tự động clone từ GitHub:
else:
    print("🌐 Chưa có repo trên Drive, đang clone từ GitHub...")
    !git clone https://github.com/vothenguyen/amodal_shape_prediction.git {LOCAL_REPO}
    %cd {LOCAL_REPO}
    !git checkout feat/d2sa

# Cài đặt các dependencies cần thiết (PyTorch đã có sẵn trong Colab)
!pip install -q timm albumentations pycocotools opencv-python tqdm

### 📦 Bước 4: Tải & Giải Nén Tập Dữ Liệu D2SA Vào SSD Cục Bộ Colab
> **Lưu ý tối ưu tốc độ:** Dữ liệu ảnh được giải nén vào ổ SSD siêu tốc của Colab (`/content/amodal_shape_prediction/data/D2SA`), còn checkpoints và logs được ghi trực tiếp vào Google Drive. Nhờ vậy tốc độ đọc ảnh đạt tối đa mà kết quả huấn luyện vẫn được sao lưu vĩnh viễn trên Drive.

In [ ]:
%%bash
mkdir -p data/D2SA
cd data/D2SA

# 1. Tải và giải nén Annotations (15.3 MB)
if [ ! -f "D2S_amodal_training_rot0.json" ]; then
    echo "📥 Đang tải Annotations D2SA..."
    curl -L -o "d2s_amodal_annotations_v1.tar.xz" \
        "https://www.mydrive.ch/shares/39000/993e79a47832a8ea7208a14d8b277c35/download/420938643-1629954673/d2s_amodal_annotations_v1.tar.xz" --progress-bar
    echo "📦 Giải nén Annotations..."
    tar -xf d2s_amodal_annotations_v1.tar.xz
    rm -f d2s_amodal_annotations_v1.tar.xz
fi

# 2. Tải và giải nén Images (4.01 GB)
if [ ! -d "images" ] || [ $(ls -1 images 2>/dev/null | wc -l) -lt 20000 ]; then
    echo "📥 Đang tải Images D2SA (~4 GB, mất khoảng 1-2 phút)..."
    curl -L -o "d2s_amodal_images_v1.tar.xz" \
        -H "User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36" \
        -H "Referer: https://www.mvtec.com/research-teaching/datasets/mvtec-d2s/downloads" \
        --connect-timeout 30 --max-time 7200 --progress-bar \
        "https://www.mydrive.ch/shares/39000/993e79a47832a8ea7208a14d8b277c35/download/420939124-1629954676/d2s_amodal_images_v1.tar.xz"
    echo "📦 Giải nén Images (22,562 ảnh)..."
    tar -xf d2s_amodal_images_v1.tar.xz
    rm -f d2s_amodal_images_v1.tar.xz
fi

echo "✅ Dữ liệu D2SA đã sẵn sàng!"

### 🔍 Bước 5: Kiểm Tra Tính Toàn Vẹn Dataset
Xác minh toàn bộ 3 file JSON annotations và 22,562 ảnh JPG.

In [ ]:
!python scripts/verify_d2sa_dataset.py

### 🧪 Bước 6 (Tùy chọn): Smoke Test Nhanh (1 Epoch trên 50 Mẫu)
Chạy thử nhanh 1 cấu hình để kiểm tra luồng nạp dữ liệu, forward pass, backward pass và tính metrics.

In [ ]:
!python scripts/run_ablation_d2sa.py \
    --rows 4 \
    --mode all \
    --epochs 1 \
    --subset-size 50 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 4 \
    --device cuda \
    --drive-dir {DRIVE_DIR}

### 🏆 Bước 7: CHẠY 1 LẦN ĐẦY ĐỦ TOÀN BỘ 6 CẤU HÌNH (TABLE V) TRÊN A100 (40GB VRAM)

Lệnh dưới đây sẽ tự động huấn luyện tuần tự cả **6 cấu hình Ablation (Row 1 -> Row 6)**:
- **Epochs:** 30 epochs mỗi cấu hình
- **Batch size:** 16, **Accumulation steps:** 1 (Effective batch = 16, tốn ~12.5GB / 40GB VRAM)
- **Num workers:** 8 (Tối đa băng thông CPU)
- **Lưu trữ:** Mọi checkpoint, log và file metrics được đồng bộ tức thì vào Google Drive
- **Đánh giá:** Tự động đánh giá trên toàn bộ tập Validation (15,654 mẫu) và tổng hợp Table V

In [ ]:
!python scripts/run_ablation_d2sa.py \
    --rows 1 2 3 4 5 6 \
    --mode all \
    --epochs 30 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 8 \
    --device cuda \
    --drive-dir {DRIVE_DIR}

### 🔄 Bước 8: Tiếp Tục Huấn Luyện Khi Bị Gián Đoạn (Resume)
Nếu chẳng may Colab bị hết lượt hoặc ngắt kết nối giữa chừng, bạn chỉ cần nạp lại Drive và chạy lệnh này để tiếp tục chính xác từ epoch bị gián đoạn:

In [ ]:
# Ví dụ: Giả sử Row 4 bị dừng ở Epoch 12, tiếp tục chạy đến Epoch 30:
# !python scripts/run_ablation_d2sa.py \
#     --rows 4 \
#     --mode all \
#     --epochs 30 \
#     --batch-size 16 \
#     --accumulation-steps 1 \
#     --num-workers 8 \
#     --resume-epoch 12 \
#     --drive-dir {DRIVE_DIR}

### 📊 Bước 9: Xem Bảng Tổng Hợp Kết Quả Table V
Hiển thị bảng kết quả so sánh giữa 6 cấu hình sau khi hoàn tất.

In [ ]:
from IPython.display import display, Markdown
import os

summary_path = f"{DRIVE_DIR}/results/d2sa/table_v_summary.md"
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f"File bảng tổng hợp chưa tồn tại tại: {summary_path}")